# 🏠🚗 AI Prediction Dashboard — Model Training
### Trains both House Price and Car Price models in one notebook
Run all cells top to bottom. Both `.pkl` files will be saved to the `models/` folder.

---
## 📦 Install & Import

In [1]:
!pip install pandas scikit-learn xgboost lightgbm joblib flask numpy --quiet

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib
import os

# Folder where .pkl files will be saved
MODELS_DIR = r"C:\Users\DELL\Desktop\CLG\Internship\ML\Prediciton\models"
os.makedirs(MODELS_DIR, exist_ok=True)

print("✅ All libraries imported")
print(f"📁 Models will be saved to: {MODELS_DIR}")

✅ All libraries imported
📁 Models will be saved to: C:\Users\DELL\Desktop\CLG\Internship\ML\Prediciton\models


---
## 🏠 MODULE 1 — House Price Prediction

### 1.1 Load Dataset

In [3]:
HOUSE_CSV = r"C:\Users\DELL\Desktop\CLG\Internship\ML\Prediciton\notebooks\train.csv"

house_df = pd.read_csv(HOUSE_CSV)
print(f"Shape: {house_df.shape}")
house_df.head()

Shape: (1460, 18)


,OverallQual,GrLivArea,GarageCars,GarageArea,TotalBsmtSF,1stFlrSF,FullBath,TotRmsAbvGrd,YearBuilt,YearRemodAdd,SalePrice,MSZoning,Utilities,BldgType,Heating,KitchenQual,SaleCondition,LandSlope
0,6,1457,3,628,1077,867,2,8,1952,1968,309505,RL,AllPub,1Fam,GasA,TA,Abnorml,Gtl
1,10,2155,2,376,1488,1406,2,12,1947,1955,409862,RL,AllPub,1Fam,GasA,Gd,Abnorml,Gtl
2,8,2132,3,638,1585,1118,1,11,1968,1984,367139,RL,AllPub,1Fam,GasA,TA,Normal,Gtl
3,7,1794,3,619,1482,1006,2,10,2004,2027,352999,RL,AllPub,1Fam,GasA,Gd,Family,Gtl
4,5,1177,2,370,787,524,3,6,1989,2009,248829,RL,AllPub,1Fam,GasA,Gd,Normal,Gtl


### 1.2 Feature Selection (correlation > 0.50 with SalePrice)

In [4]:
# Numeric features selected by correlation (exact notebook logic)
df1 = house_df.select_dtypes(include=["int64", "float64"])
corr = df1.corr()["SalePrice"]
important_num_cols = list(corr[(corr > 0.50) | (corr < -0.50)].index)

# Categorical features (exact notebook selection)
cat_cols_house = ["MSZoning", "Utilities", "BldgType", "Heating",
                  "KitchenQual", "SaleCondition", "LandSlope"]

important_cols = important_num_cols + cat_cols_house
house_df = house_df[important_cols]

print("Numeric features selected:", important_num_cols)
print("Categorical features:", cat_cols_house)

Numeric features selected: ['OverallQual', 'GrLivArea', 'TotalBsmtSF', '1stFlrSF', 'TotRmsAbvGrd', 'SalePrice']
Categorical features: ['MSZoning', 'Utilities', 'BldgType', 'Heating', 'KitchenQual', 'SaleCondition', 'LandSlope']


### 1.3 Preprocess

In [5]:
# X / y split
X_house = house_df.drop("SalePrice", axis=1)
y_house = house_df["SalePrice"]

# One-hot encode categorical columns
X_house = pd.get_dummies(X_house, columns=cat_cols_house)

# Scale numeric columns only
num_cols_house = [c for c in important_num_cols if c != "SalePrice"]
house_scaler = StandardScaler()
X_house[num_cols_house] = house_scaler.fit_transform(X_house[num_cols_house])

print(f"Total features after encoding: {X_house.shape[1]}")
print("Feature columns:", X_house.columns.tolist())

Total features after encoding: 36
Feature columns: ['OverallQual', 'GrLivArea', 'TotalBsmtSF', '1stFlrSF', 'TotRmsAbvGrd', 'MSZoning_C (all)', 'MSZoning_FV', 'MSZoning_RH', 'MSZoning_RL', 'MSZoning_RM', 'Utilities_AllPub', 'Utilities_NoSeWa', 'BldgType_1Fam', 'BldgType_2fmCon', 'BldgType_Duplex', 'BldgType_Twnhs', 'BldgType_TwnhsE', 'Heating_GasA', 'Heating_GasW', 'Heating_Grav', 'Heating_OthW', 'Heating_Wall', 'KitchenQual_Ex', 'KitchenQual_Fa', 'KitchenQual_Gd', 'KitchenQual_Po', 'KitchenQual_TA', 'SaleCondition_Abnorml', 'SaleCondition_AdjLand', 'SaleCondition_Alloca', 'SaleCondition_Family', 'SaleCondition_Normal', 'SaleCondition_Partial', 'LandSlope_Gtl', 'LandSlope_Mod', 'LandSlope_Sev']


### 1.4 Train / Test Split

In [6]:
X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    X_house, y_house, test_size=0.2, random_state=121
)
print(f"Train: {X_h_train.shape[0]} rows  |  Test: {X_h_test.shape[0]} rows")

Train: 1168 rows  |  Test: 292 rows


### 1.5 Train GradientBoostingRegressor

In [7]:
house_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
house_model.fit(X_h_train, y_h_train)

# Evaluate
y_pred_house = house_model.predict(X_h_test)
print(f"R²   : {r2_score(y_h_test, y_pred_house):.4f}")
print(f"MAE  : ${mean_absolute_error(y_h_test, y_pred_house):,.0f}")
print(f"RMSE : ${np.sqrt(mean_squared_error(y_h_test, y_pred_house)):,.0f}")

R²   : 0.8764
MAE  : $18,809
RMSE : $23,241


### 1.6 Save House Model

In [8]:
house_artifacts = {
    "model":           house_model,
    "scaler":          house_scaler,
    "feature_columns": X_house.columns.tolist(),
    "numeric_columns": num_cols_house,
    "cat_cols":        cat_cols_house,
}

house_pkl_path = os.path.join(MODELS_DIR, "house_model.pkl")
joblib.dump(house_artifacts, house_pkl_path)

print(f"✅ house_model.pkl saved → {house_pkl_path}")
print(f"   Features : {len(X_house.columns)}")
print(f"   File size: {os.path.getsize(house_pkl_path) / 1024:.1f} KB")

✅ house_model.pkl saved → C:\Users\DELL\Desktop\CLG\Internship\ML\Prediciton\models\house_model.pkl
   Features : 36
   File size: 136.6 KB


---
## 🚗 MODULE 2 — Car Price Prediction

### 2.1 Load Dataset

In [9]:
CAR_CSV = r"C:\Users\DELL\Desktop\CLG\Internship\ML\Prediciton\notebooks\car_price_dataset.csv"

car_df = pd.read_csv(CAR_CSV)
print(f"Shape: {car_df.shape}")
car_df.head()

Shape: (5000, 16)


,Year,Mileage_km,Engine_CC,Horsepower,Num_Owners,Doors,Seats,Service_History,Accidents_Reported,Insurance_Valid,Brand,Fuel_Type,Transmission,Body_Type,Condition,Price_USD
0,2021,58891.0,1600,125,1,4,5,1,0,1,Volkswagen,Diesel,Manual,Sedan,Good,11220.30
1,2022,19846.0,1500,122,2,4,4,1,0,1,Mercedes,Petrol,Manual,SUV,Fair,7187.11
2,2021,54562.0,1800,132,2,4,5,1,1,1,Hyundai,Diesel,Automatic,Coupe,Fair,5455.30
3,2023,18141.0,1600,140,1,4,7,1,0,1,Toyota,Diesel,Automatic,SUV,Good,9609.19
4,2008,193764.0,1800,135,2,4,5,0,1,0,BMW,Petrol,Automatic,Coupe,Good,2389.53


### 2.2 Handle Missing Values

In [10]:
# Median imputation on columns that may have nulls (exact notebook logic)
num_imputer = SimpleImputer(strategy="median")
car_df["Mileage_km"] = num_imputer.fit_transform(car_df[["Mileage_km"]]).ravel()
car_df["Engine_CC"]  = num_imputer.fit_transform(car_df[["Engine_CC"]]).ravel()

print(f"Missing values remaining: {car_df.isnull().sum().sum()}")

Missing values remaining: 0


### 2.3 Feature Engineering

In [11]:
# Derived features (exact notebook logic)
car_df["Car_Age"]         = 2025 - car_df["Year"]
car_df["Mileage_per_Year"] = car_df["Mileage_km"] / (car_df["Car_Age"] + 1)

print("New features: Car_Age, Mileage_per_Year")
car_df[["Year", "Car_Age", "Mileage_km", "Mileage_per_Year"]].head()

New features: Car_Age, Mileage_per_Year


,Year,Car_Age,Mileage_km,Mileage_per_Year
0,2021,4,58891.0,11778.200000
1,2022,3,19846.0,4961.500000
2,2021,4,54562.0,10912.400000
3,2023,2,18141.0,6047.000000
4,2008,17,193764.0,10764.666667


### 2.4 Feature Selection

In [12]:
important_num_cols_car = [
    "Car_Age", "Mileage_km", "Mileage_per_Year",
    "Engine_CC", "Horsepower", "Num_Owners",
    "Doors", "Seats", "Service_History",
    "Accidents_Reported", "Insurance_Valid"
]
cat_cols_car = ["Brand", "Fuel_Type", "Transmission", "Body_Type", "Condition"]

print(f"Numeric  : {len(important_num_cols_car)} features")
print(f"Categorical: {len(cat_cols_car)} features")

Numeric  : 11 features
Categorical: 5 features


### 2.5 Encode & Scale

In [13]:
# One-hot encode
df_encoded = pd.get_dummies(
    car_df[cat_cols_car + important_num_cols_car + ["Price_USD"]],
    columns=cat_cols_car
)

X_car = df_encoded.drop("Price_USD", axis=1)
y_car = df_encoded["Price_USD"]

# Train / test split
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_car, y_car, test_size=0.2, random_state=42
)

# Scale entire feature matrix (exact notebook logic)
car_scaler = StandardScaler()
X_c_train_scaled = car_scaler.fit_transform(X_c_train)
X_c_test_scaled  = car_scaler.transform(X_c_test)

print(f"Total features after encoding: {X_car.shape[1]}")
print(f"Train: {X_c_train.shape[0]} rows  |  Test: {X_c_test.shape[0]} rows")

Total features after encoding: 38
Train: 4000 rows  |  Test: 1000 rows


### 2.6 Train GradientBoostingRegressor

In [14]:
car_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, random_state=42)
car_model.fit(X_c_train_scaled, y_c_train)

# Evaluate
y_pred_car = car_model.predict(X_c_test_scaled)
print(f"R²   : {r2_score(y_c_test, y_pred_car):.4f}")
print(f"MAE  : ${mean_absolute_error(y_c_test, y_pred_car):,.0f}")
print(f"RMSE : ${np.sqrt(mean_squared_error(y_c_test, y_pred_car)):,.0f}")

R²   : 0.8593
MAE  : $1,403
RMSE : $1,788


### 2.7 Save Car Model

In [15]:
car_artifacts = {
    "model":               car_model,
    "scaler":              car_scaler,
    "feature_columns":     X_car.columns.tolist(),
    "numeric_columns":     important_num_cols_car,
    "cat_cols":            cat_cols_car,
    "brand_values":        sorted(car_df["Brand"].dropna().unique().tolist()),
    "fuel_type_values":    sorted(car_df["Fuel_Type"].dropna().unique().tolist()),
    "transmission_values": sorted(car_df["Transmission"].dropna().unique().tolist()),
    "body_type_values":    sorted(car_df["Body_Type"].dropna().unique().tolist()),
    "condition_values":    ["New", "Like New", "Good", "Fair", "Poor"],
}

car_pkl_path = os.path.join(MODELS_DIR, "car_model.pkl")
joblib.dump(car_artifacts, car_pkl_path)

print(f"✅ car_model.pkl saved → {car_pkl_path}")
print(f"   Features : {len(X_car.columns)}")
print(f"   File size: {os.path.getsize(car_pkl_path) / 1024:.1f} KB")

✅ car_model.pkl saved → C:\Users\DELL\Desktop\CLG\Internship\ML\Prediciton\models\car_model.pkl
   Features : 38
   File size: 267.4 KB


---
## ✅ Final Verification

In [16]:
# Reload both and confirm
h = joblib.load(house_pkl_path)
c = joblib.load(car_pkl_path)

print("=" * 50)
print("🏠 HOUSE MODEL")
print(f"   Model      : {type(h['model']).__name__}")
print(f"   Features   : {len(h['feature_columns'])}")
print(f"   Numeric    : {h['numeric_columns']}")
print()
print("🚗 CAR MODEL")
print(f"   Model      : {type(c['model']).__name__}")
print(f"   Features   : {len(c['feature_columns'])}")
print(f"   Numeric    : {c['numeric_columns']}")
print(f"   Brands     : {c['brand_values']}")
print("=" * 50)
print("✅ Both models ready. Run app.py to launch the dashboard.")

🏠 HOUSE MODEL
   Model      : GradientBoostingRegressor
   Features   : 36
   Numeric    : ['OverallQual', 'GrLivArea', 'TotalBsmtSF', '1stFlrSF', 'TotRmsAbvGrd']

🚗 CAR MODEL
   Model      : GradientBoostingRegressor
   Features   : 38
   Numeric    : ['Car_Age', 'Mileage_km', 'Mileage_per_Year', 'Engine_CC', 'Horsepower', 'Num_Owners', 'Doors', 'Seats', 'Service_History', 'Accidents_Reported', 'Insurance_Valid']
   Brands     : ['Audi', 'BMW', 'Ford', 'Honda', 'Hyundai', 'Kia', 'Mercedes', 'Nissan', 'Toyota', 'Volkswagen']
✅ Both models ready. Run app.py to launch the dashboard.
